# Imports

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm
from ucimlrepo import fetch_ucirepo 
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load Data

In [43]:

  
# fetch dataset 
magic_gamma_telescope = fetch_ucirepo(id=159) 
  
# data (as pandas dataframes) 
X = magic_gamma_telescope.data.features 
y = magic_gamma_telescope.data.targets 
  
# metadata 
print(magic_gamma_telescope.metadata) 
  
# variable information 
print(magic_gamma_telescope.variables) 


{'uci_id': 159, 'name': 'MAGIC Gamma Telescope', 'repository_url': 'https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope', 'data_url': 'https://archive.ics.uci.edu/static/public/159/data.csv', 'abstract': 'Data are MC generated to simulate registration of high energy gamma particles in an atmospheric Cherenkov telescope', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 19020, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2004, 'last_updated': 'Tue Dec 19 2023', 'dataset_doi': '10.24432/C52C8B', 'creators': ['R. Bock'], 'intro_paper': None, 'additional_info': {'summary': "The data are MC generated (see below) to simulate registration of high energy gamma particles in a ground-based atmospheric Cherenkov gamma telescope using the imaging technique. Cherenkov gamm

In [44]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 19020 entries, 0 to 19019
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   fLength   19020 non-null  float64
 1   fWidth    19020 non-null  float64
 2   fSize     19020 non-null  float64
 3   fConc     19020 non-null  float64
 4   fConc1    19020 non-null  float64
 5   fAsym     19020 non-null  float64
 6   fM3Long   19020 non-null  float64
 7   fM3Trans  19020 non-null  float64
 8   fAlpha    19020 non-null  float64
 9   fDist     19020 non-null  float64
dtypes: float64(10)
memory usage: 1.5 MB


In [45]:
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 19020 entries, 0 to 19019
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   class   19020 non-null  str  
dtypes: str(1)
memory usage: 148.7 KB


# Label Encoding

In [46]:
y

,class
0,g
1,g
2,g
3,g
4,g
...,...
19015,h
19016,h
19017,h
19018,h


In [47]:
le = LabelEncoder()
y = pd.Series(
    le.fit_transform(y),
    index=y.index,
    name="Severity"
)
y.value_counts()
y

c:\Users\chris\Documents\GitHub\ML_DL\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0        0
1        0
2        0
3        0
4        0
        ..
19015    1
19016    1
19017    1
19018    1
19019    1
Name: Severity, Length: 19020, dtype: int64

# Split

In [48]:
X_train, X_test_val, y_train, y_test_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_test, X_val, y_test, y_val = train_test_split(
    X_test_val,
    y_test_val,
    test_size=0.5,   # 0.5 of 0.2 = 0.1 of the full dataset
    random_state=42,
    stratify=y_test_val
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 15216
Validation size: 1902
Test size: 1902


# Standardize

In [49]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [50]:
X_train

array([[ 0.91871221, -0.80967508, -0.90694745, ..., -0.16184865,
         1.16310154, -0.13654401],
       [-0.85201165, -0.54005485, -0.35542251, ...,  0.42658507,
        -0.46788101, -0.8159116 ],
       [-0.94695582, -0.61226917, -1.32329679, ...,  0.5133034 ,
         0.74142789, -2.12586548],
       ...,
       [-0.72886562, -0.21700808, -0.53579811, ..., -0.65168244,
        -0.5287884 , -0.48680656],
       [ 1.28603108,  0.12157189,  0.68417761, ..., -1.00020624,
        -0.8759073 ,  0.63787193],
       [-0.81539435, -0.80133591, -1.42770243, ...,  0.12050591,
         0.60652129, -0.16293072]], shape=(15216, 10))

# Dataloader

In [51]:
# Convert data to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)



y_train = torch.tensor(y_train.squeeze().to_numpy(), dtype=torch.long)
y_val = torch.tensor(y_val.squeeze().to_numpy(), dtype=torch.long)
y_test = torch.tensor(y_test.squeeze().to_numpy(), dtype=torch.long)


# Create datasets and dataloaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [52]:
next(iter(train_loader))

[tensor([[-0.6765, -0.4285, -0.5507,  0.4426,  0.2663, -0.3269,  0.0451,  0.3782,
          -0.9773, -1.0657],
         [ 2.7731,  1.4111,  0.7130, -1.2345, -1.1733, -4.3814,  2.4297,  0.5677,
          -0.1753, -0.0541],
         [-0.8324, -0.3490, -0.6686,  0.7712,  0.5965,  0.3960,  0.1419,  0.2864,
           1.9885, -0.5000],
         [-0.7248, -0.6231, -1.0508,  0.8497,  0.6700,  0.4638,  0.1630,  0.4697,
           2.1061, -2.1362],
         [-0.5637, -0.6706, -0.9163,  0.6953,  0.6455,  0.2491,  0.1472, -0.5845,
           1.4910, -0.3560],
         [-0.9421, -0.6294, -1.3895,  2.2297,  1.8946, -0.1737, -0.2756,  0.5495,
           1.2656, -0.4063],
         [-0.5705, -0.0757,  0.2727, -0.2072, -0.3533,  0.1923,  0.3130, -0.8795,
           0.1571, -0.5265],
         [-0.5858, -0.2958, -0.5946,  0.2674,  0.0105,  0.5041,  0.2080, -0.4958,
          -0.8825, -1.8412],
         [-0.0779, -0.0078,  0.0900, -1.0955, -1.0164, -0.5041,  0.4222,  0.5193,
           1.9162,  0.6246],
 

# Model

In [53]:
class advanced_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()

        # A simple network with one hidden layer and dropout
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [54]:
advanced_model = advanced_MLP(10, 32, 2, 0.3)

In [55]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
advanced_model_optimizer = optim.Adam(advanced_model.parameters(),lr=0.01)

In [56]:

num_epochs = 100
patience = 10  # Stop if validation accuracy does not improve for 10 epochs


In [57]:

best_val_accuracy = 0.0
best_model_state = {
    key: value.detach().clone()
    for key, value in advanced_model.state_dict().items()
}
epochs_without_improvement = 0

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    advanced_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in train_progress:
        # Forward pass
        train_outputs = advanced_model(batch_X)
        loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        advanced_model_optimizer.zero_grad()
        loss.backward()
        advanced_model_optimizer.step()

        # Collect training statistics
        train_loss_sum += loss.item() * batch_X.size(0)
        predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        train_progress.set_postfix({
            "batch_loss": f"{loss.item():.4f}"
        })

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    advanced_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = advanced_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += loss.item() * batch_X.size(0)
            predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # -----------------------------
    # Early stopping
    # -----------------------------
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().clone()
            for key, value in advanced_model.state_dict().items()
        }
        epochs_without_improvement = 0
        print("Validation accuracy improved, saving model state.")
        print(f"Patience: {epochs_without_improvement}/{patience}")
    else:
        epochs_without_improvement += 1
        print(f"Patience: {epochs_without_improvement}/{patience}")
    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break



Epoch [1/100] | Train Loss: 0.4121 | Train Acc: 0.8210 | Val Loss: 0.3825 | Val Acc: 0.8612
Validation accuracy improved, saving model state.
Patience: 0/10


Epoch [2/100] | Train Loss: 0.3771 | Train Acc: 0.8441 | Val Loss: 0.3002 | Val Acc: 0.8586
Patience: 1/10


Epoch [3/100] | Train Loss: 0.3755 | Train Acc: 0.8421 | Val Loss: 0.5721 | Val Acc: 0.8633
Validation accuracy improved, saving model state.
Patience: 0/10


Epoch [4/100] | Train Loss: 0.3648 | Train Acc: 0.8481 | Val Loss: 0.3930 | Val Acc: 0.8749
Validation accuracy improved, saving model state.
Patience: 0/10


Epoch [5/100] | Train Loss: 0.3691 | Train Acc: 0.8476 | Val Loss: 0.3072 | Val Acc: 0.8686
Patience: 1/10


...

Epoch [24/100] | Train Loss: 0.3586 | Train Acc: 0.8513 | Val Loss: 0.1816 | Val Acc: 0.8812
Patience: 6/10


Epoch [25/100] | Train Loss: 0.3549 | Train Acc: 0.8552 | Val Loss: 0.1568 | Val Acc: 0.8812
Patience: 7/10


Epoch [26/100] | Train Loss: 0.3551 | Train Acc: 0.8526 | Val Loss: 0.4177 | Val Acc: 0.8675
Patience: 8/10


Epoch [27/100] | Train Loss: 0.3574 | Train Acc: 0.8502 | Val Loss: 0.2758 | Val Acc: 0.8801
Patience: 9/10


Epoch [28/100] | Train Loss: 0.3532 | Train Acc: 0.8520 | Val Loss: 0.2916 | Val Acc: 0.8822
Patience: 10/10

Early stopping triggered.


In [58]:

# Load the best model
advanced_model.load_state_dict(best_model_state)

advanced_model.eval()
test_correct = 0
test_total = 0

all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        test_outputs = advanced_model(batch_X)
        test_predictions = torch.argmax(test_outputs, dim=1)

        # Update accuracy statistics
        test_correct += (test_predictions == batch_y).sum().item()
        test_total += batch_y.size(0)

        # Store predictions and true labels
        all_y_true.extend(batch_y.numpy())
        all_y_pred.extend(test_predictions.numpy())

test_accuracy = test_correct / test_total
print("\nFinal Test Accuracy:", test_accuracy)

# Print classification report
print("\nClassification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=magic_gamma_telescope.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(all_y_true, all_y_pred))


Final Test Accuracy: 0.8654048370136698

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.96      0.90      1233
           1       0.90      0.69      0.78       669

    accuracy                           0.87      1902
   macro avg       0.88      0.83      0.84      1902
weighted avg       0.87      0.87      0.86      1902

Confusion Matrix:
[[1184   49]
 [ 207  462]]
